In [1]:
nsga2_code = '''import numpy as np
import sys
sys.path.insert(0, '.')
from ga.ga_core import tour_cost

from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.core.problem import Problem
from pymoo.core.sampling import Sampling
from pymoo.operators.crossover.ox import OrderCrossover
from pymoo.operators.mutation.inversion import InversionMutation
from pymoo.optimize import minimize

class MOTSPProblem(Problem):
    def __init__(self, D, T, C):
        self.D, self.T, self.C = D, T, C
        n = D.shape[0]
        super().__init__(n_var=n, n_obj=3, vtype=int)

    def _evaluate(self, X, out, *args, **kwargs):
        F = np.array([
            [tour_cost(x, self.D),
             tour_cost(x, self.T),
             tour_cost(x, self.C)]
            for x in X
        ])
        out["F"] = F

class PermutationSampling(Sampling):
    def _do(self, problem, n_samples, **kwargs):
        X = np.array([np.random.permutation(problem.n_var)
                      for _ in range(n_samples)])
        return X

def run_nsga2(D, T, C, N=100, G_max=500, seed=0):
    np.random.seed(seed)
    problem = MOTSPProblem(D, T, C)
    algo = NSGA2(
        pop_size=N,
        sampling=PermutationSampling(),
        crossover=OrderCrossover(),
        mutation=InversionMutation(),
        eliminate_duplicates=False
    )
    res = minimize(problem, algo,
                   termination=("n_gen", G_max),
                   seed=seed, verbose=False)

    archive_objs = res.F.tolist() if res.F is not None else []
    log = [{"g": G_max, "archive_size": len(archive_objs)}]
    return archive_objs, log
'''

os.makedirs('baselines', exist_ok=True)
with open('baselines/nsga2_baseline.py', 'w', encoding='utf-8') as f:
    f.write(nsga2_code)
print('baselines/nsga2_baseline.py saved.')

baselines/nsga2_baseline.py saved.


In [2]:
moead_code = '''import numpy as np
import sys
sys.path.insert(0, ".")
from ga.ga_core import (init_population, evaluate_population,
                         normalise_objectives, compute_ideal_point,
                         compute_all_fitness, run_one_generation)
from rl.pareto import ParetoArchive

def generate_weight_vectors(N, n_obj=3):
    """
    Generate N uniformly distributed weight vectors over the simplex.
    Uses Dirichlet sampling with concentration=1 for uniformity.
    """
    rng = np.random.default_rng(0)          # fixed for reproducibility
    W = rng.dirichlet(np.ones(n_obj), size=N)
    W = np.clip(W, 0.05, 0.90)
    W /= W.sum(axis=1, keepdims=True)
    return W                                 # shape (N, 3)

def run_moead(D, T, C, N=100, G_max=500,
              pc=0.9, pm=0.02, elitism=2,
              archive_max=200, seed=0):
    """
    MOEA/D with Tchebycheff decomposition and static weight vectors.
    Each individual is evaluated under its own fixed weight vector.
    The population-level fitness is the vector of individual Tchebycheff values.
    """
    np.random.seed(seed)
    rng = np.random.default_rng(seed)
    n_cities = D.shape[0]

    W       = generate_weight_vectors(N)     # fixed weights — never updated
    archive = ParetoArchive(max_size=archive_max)
    pop     = init_population(N, n_cities, D, seed=seed)
    log     = []

    for g in range(1, G_max + 1):
        obj_costs = evaluate_population(pop, D, T, C)
        f_norm    = normalise_objectives(obj_costs)
        z_star    = compute_ideal_point(f_norm)
        archive.update(pop, obj_costs)

        # Each individual evaluated under its OWN weight vector
        fitness = np.array([
            np.max(W[i] * np.abs(f_norm[i] - z_star))
            for i in range(N)
        ])

        # Use mean weight for crossover/mutation operators
        w_mean = W.mean(axis=0)
        pop = run_one_generation(pop, fitness, w_mean, D, T, C,
                                  pc=pc, pm=pm, elitism=elitism, rng=rng)

        log.append({"g": g, "archive_size": len(archive),
                    "f_best": float(fitness.min())})

    archive_objs = archive.get_objectives().tolist()
    return archive_objs, log
'''

with open('baselines/moead_baseline.py', 'w', encoding='utf-8') as f:
    f.write(moead_code)
print('baselines/moead_baseline.py saved.')

baselines/moead_baseline.py saved.


In [3]:
ablations_code = '''import numpy as np
import sys
sys.path.insert(0, ".")
from ga.ga_core import (init_population, evaluate_population,
                         normalise_objectives, compute_ideal_point,
                         compute_all_fitness, run_one_generation)
from rl.pareto import ParetoArchive

def run_ga_static(D, T, C, N=100, G_max=500,
                  pc=0.9, pm=0.02, elitism=2,
                  archive_max=200, seed=0):
    """
    GA with static equal weights (1/3, 1/3, 1/3) — no RL controller.
    Ablation baseline: proves RL adaptation adds value.
    """
    np.random.seed(seed)
    rng     = np.random.default_rng(seed)
    n       = D.shape[0]
    w       = np.array([1/3, 1/3, 1/3])    # FIXED — never changes
    archive = ParetoArchive(max_size=archive_max)
    pop     = init_population(N, n, D, seed=seed)
    log     = []

    for g in range(1, G_max + 1):
        obj_costs = evaluate_population(pop, D, T, C)
        f_norm    = normalise_objectives(obj_costs)
        z_star    = compute_ideal_point(f_norm)
        fitness   = compute_all_fitness(f_norm, w, z_star)
        archive.update(pop, obj_costs)
        pop = run_one_generation(pop, fitness, w, D, T, C,
                                  pc=pc, pm=pm, elitism=elitism, rng=rng)
        log.append({"g": g, "archive_size": len(archive),
                    "f_best": float(fitness.min())})

    return archive.get_objectives().tolist(), log


def run_ga_random(D, T, C, N=100, G_max=500, delta_g=5,
                  pc=0.9, pm=0.02, elitism=2,
                  archive_max=200, seed=0):
    """
    GA with randomly perturbed weights every delta_g generations — no RL.
    Ablation baseline: proves intelligent adaptation matters, not just variation.
    """
    np.random.seed(seed)
    rng     = np.random.default_rng(seed)
    n       = D.shape[0]
    w       = np.array([1/3, 1/3, 1/3])
    archive = ParetoArchive(max_size=archive_max)
    pop     = init_population(N, n, D, seed=seed)
    log     = []

    for g in range(1, G_max + 1):
        obj_costs = evaluate_population(pop, D, T, C)
        f_norm    = normalise_objectives(obj_costs)
        z_star    = compute_ideal_point(f_norm)
        fitness   = compute_all_fitness(f_norm, w, z_star)
        archive.update(pop, obj_costs)
        pop = run_one_generation(pop, fitness, w, D, T, C,
                                  pc=pc, pm=pm, elitism=elitism, rng=rng)

        if g % delta_g == 0:
            w = rng.dirichlet(np.ones(3))   # random — no learning
            w = np.clip(w, 0.05, 0.90)
            w /= w.sum()

        log.append({"g": g, "archive_size": len(archive),
                    "f_best": float(fitness.min())})

    return archive.get_objectives().tolist(), log
'''

with open('baselines/ablations.py', 'w', encoding='utf-8') as f:
    f.write(ablations_code)
print('baselines/ablations.py saved.')

baselines/ablations.py saved.


In [4]:
from rl_aga.rl_aga_main       import run_rl_aga
from baselines.nsga2_baseline  import run_nsga2
from baselines.moead_baseline  import run_moead
from baselines.ablations       import run_ga_static, run_ga_random

print('Smoke testing all 5 algorithms on eil51 (N=30, G=30)...\n')

configs = [
    ('RL-AGA',      lambda: run_rl_aga(D, T, C, N=30, G_max=30, seed=0)),
    ('NSGA-II',     lambda: run_nsga2(D, T, C,   N=30, G_max=30, seed=0)),
    ('MOEA/D',      lambda: run_moead(D, T, C,   N=30, G_max=30, seed=0)),
    ('GA-Static',   lambda: run_ga_static(D, T, C, N=30, G_max=30, seed=0)),
    ('GA-Random',   lambda: run_ga_random(D, T, C, N=30, G_max=30, seed=0)),
]

for name, fn in configs:
    t0 = time.time()
    result = fn()

    # All return (archive_objs, log)
    if name == 'RL-AGA':
        archive_obj, log, _ = result
        arc_objs = archive_obj.get_objectives()
        arc_size = len(archive_obj)
    else:
        arc_objs_list, log = result
        arc_size = len(arc_objs_list)

    elapsed = time.time() - t0
    print(f'{name:<12} archive={arc_size:>3}  '
          f'log_len={len(log):>3}  time={elapsed:.1f}s')
    assert arc_size > 0,   f'{name}: empty archive!'
    assert len(log)  > 0,  f'{name}: empty log!'

print('\nAll algorithms smoke test PASSED.')

Smoke testing all 5 algorithms on eil51 (N=30, G=30)...



NameError: name 'time' is not defined

In [5]:
nsga2_code = '''import numpy as np
import sys
sys.path.insert(0, ".")

from ga.ga_core  import (init_population, evaluate_population,
                          order_crossover, swap_mutation, two_opt_fast)
from rl.pareto   import ParetoArchive

# ── Non-dominated sorting (NSGA-II core) ──────────────────────────────────────

def fast_non_dominated_sort(obj_costs):
    """
    Assign Pareto rank to each individual.
    Rank 0 = non-dominated (best). O(N^2).
    """
    N = len(obj_costs)
    dom_count  = np.zeros(N, dtype=int)   # how many solutions dominate i
    dom_set    = [[] for _ in range(N)]   # solutions that i dominates

    for i in range(N):
        for j in range(i + 1, N):
            fi, fj = obj_costs[i], obj_costs[j]
            if np.all(fi <= fj) and np.any(fi < fj):   # i dominates j
                dom_set[i].append(j)
                dom_count[j] += 1
            elif np.all(fj <= fi) and np.any(fj < fi): # j dominates i
                dom_set[j].append(i)
                dom_count[i] += 1

    ranks = np.full(N, -1, dtype=int)
    front = [i for i in range(N) if dom_count[i] == 0]
    rank  = 0
    while front:
        for i in front:
            ranks[i] = rank
        next_front = []
        for i in front:
            for j in dom_set[i]:
                dom_count[j] -= 1
                if dom_count[j] == 0:
                    next_front.append(j)
        front = next_front
        rank += 1
    return ranks

def crowding_distance(obj_costs_front):
    """Crowding distance for one Pareto front."""
    n, m  = obj_costs_front.shape
    dist  = np.zeros(n)
    for k in range(m):
        idx  = np.argsort(obj_costs_front[:, k])
        dist[idx[0]] = dist[idx[-1]] = np.inf
        span = obj_costs_front[idx[-1], k] - obj_costs_front[idx[0], k]
        if span > 1e-10:
            for i in range(1, n - 1):
                dist[idx[i]] += (obj_costs_front[idx[i+1], k] -
                                  obj_costs_front[idx[i-1], k]) / span
    return dist

def nsga2_fitness(ranks, obj_costs):
    """
    Combined NSGA-II fitness: lower rank = better.
    Within same rank, higher crowding distance = better.
    Returns a scalar fitness where lower is better.
    """
    N     = len(ranks)
    cd    = np.zeros(N)
    for r in np.unique(ranks):
        idx = np.where(ranks == r)[0]
        if len(idx) > 2:
            cd[idx] = crowding_distance(obj_costs[idx])
    # fitness = rank - normalised crowding distance (lower = better)
    cd_norm = cd / (cd[np.isfinite(cd)].max() + 1e-10)
    return ranks.astype(float) - np.where(np.isinf(cd), 1.0, cd_norm)

def tournament_select(pop, fitness, rng):
    idx    = rng.choice(len(pop), 2, replace=False)
    winner = idx[np.argmin(fitness[idx])]
    return pop[winner].copy()

# ── Main NSGA-II function ──────────────────────────────────────────────────────

def run_nsga2(D, T, C, N=100, G_max=200,
              pc=0.9, pm=0.02, archive_max=200, seed=0):
    """
    NSGA-II for MC-TSP using proven GA operators and an external Pareto archive.
    Non-dominated sorting provides selection pressure.
    External archive accumulates all non-dominated solutions across all generations.
    """
    np.random.seed(seed)
    rng     = np.random.default_rng(seed)
    n       = D.shape[0]
    archive = ParetoArchive(max_size=archive_max)
    pop     = init_population(N, n, D, seed=seed)
    M_avg   = (D + T + C) / 3.0   # neutral combined matrix for 2-Opt

    log = []

    for g in range(1, G_max + 1):
        obj_costs = evaluate_population(pop, D, T, C)
        archive.update(pop, obj_costs)

        # NSGA-II selection pressure
        ranks   = fast_non_dominated_sort(obj_costs)
        fitness = nsga2_fitness(ranks, obj_costs)

        # Generate offspring
        new_pop = []
        while len(new_pop) < N:
            p1    = tournament_select(pop, fitness, rng)
            p2    = tournament_select(pop, fitness, rng)
            child = order_crossover(p1, p2, rng) if rng.random() < pc else p1.copy()
            child = swap_mutation(child, pm, rng)
            child = two_opt_fast(child, M_avg, max_passes=2)
            new_pop.append(child)
        pop = np.array(new_pop)

        log.append({"g": g, "archive_size": len(archive),
                    "f_best": float(fitness.min())})

    return archive.get_objectives().tolist(), log
'''

import os
with open('baselines/nsga2_baseline.py', 'w', encoding='utf-8') as f:
    f.write(nsga2_code)
print('baselines/nsga2_baseline.py rewritten.')

baselines/nsga2_baseline.py rewritten.


In [6]:
nsga2_code = '''import numpy as np
import sys
sys.path.insert(0, ".")

from ga.ga_core import (
    init_population,
    evaluate_population,
    order_crossover,
    swap_mutation,
    two_opt_fast
)

from rl.pareto import ParetoArchive


# ─────────────────────────────────────────────────────────────
# Pareto dominance
# ─────────────────────────────────────────────────────────────

def dominates(a, b):
    return np.all(a <= b) and np.any(a < b)


# ─────────────────────────────────────────────────────────────
# Fast non-dominated sorting
# ─────────────────────────────────────────────────────────────

def fast_non_dominated_sort(obj_costs):

    N = len(obj_costs)

    dom_count = np.zeros(N, dtype=int)
    dom_set = [[] for _ in range(N)]

    for i in range(N):
        for j in range(i + 1, N):

            fi = obj_costs[i]
            fj = obj_costs[j]

            if dominates(fi, fj):
                dom_set[i].append(j)
                dom_count[j] += 1

            elif dominates(fj, fi):
                dom_set[j].append(i)
                dom_count[i] += 1

    ranks = np.full(N, -1, dtype=int)

    front = [i for i in range(N) if dom_count[i] == 0]

    rank = 0

    while front:

        for i in front:
            ranks[i] = rank

        next_front = []

        for i in front:

            for j in dom_set[i]:

                dom_count[j] -= 1

                if dom_count[j] == 0:
                    next_front.append(j)

        front = next_front
        rank += 1

    return ranks


# ─────────────────────────────────────────────────────────────
# Crowding distance
# ─────────────────────────────────────────────────────────────

def crowding_distance(obj_costs_front):

    n, m = obj_costs_front.shape

    if n <= 2:
        return np.full(n, np.inf)

    dist = np.zeros(n)

    for k in range(m):

        idx = np.argsort(obj_costs_front[:, k])

        dist[idx[0]] = np.inf
        dist[idx[-1]] = np.inf

        span = (
            obj_costs_front[idx[-1], k]
            - obj_costs_front[idx[0], k]
        )

        if span > 1e-10:

            for i in range(1, n - 1):

                dist[idx[i]] += (
                    obj_costs_front[idx[i + 1], k]
                    - obj_costs_front[idx[i - 1], k]
                ) / span

    return dist


# ─────────────────────────────────────────────────────────────
# NSGA-II fitness
# ─────────────────────────────────────────────────────────────

def nsga2_fitness(ranks, obj_costs):

    N = len(ranks)

    cd = np.zeros(N)

    for r in np.unique(ranks):

        idx = np.where(ranks == r)[0]

        if len(idx) > 2:
            cd[idx] = crowding_distance(obj_costs[idx])

        else:
            cd[idx] = np.inf

    finite_cd = cd[np.isfinite(cd)]

    if len(finite_cd) > 0:
        cd_norm = cd / (finite_cd.max() + 1e-10)
    else:
        cd_norm = np.zeros_like(cd)

    return ranks.astype(float) - np.where(
        np.isinf(cd),
        1.0,
        cd_norm
    )


# ─────────────────────────────────────────────────────────────
# Tournament selection
# ─────────────────────────────────────────────────────────────

def tournament_select(pop, fitness, rng):

    idx = rng.choice(len(pop), 2, replace=False)

    winner = idx[np.argmin(fitness[idx])]

    return pop[winner].copy()


# ─────────────────────────────────────────────────────────────
# Main NSGA-II
# ─────────────────────────────────────────────────────────────

def run_nsga2(
    D,
    T,
    C,
    N=100,
    G_max=200,
    pc=0.9,
    pm=0.10,
    archive_max=200,
    seed=0
):

    np.random.seed(seed)

    rng = np.random.default_rng(seed)

    n = D.shape[0]

    archive = ParetoArchive(max_size=archive_max)

    pop = init_population(N, n, D, seed=seed)

    # Neutral matrix for local search
    M_avg = (D + T + C) / 3.0

    log = []

    for g in range(1, G_max + 1):

        # -----------------------------------------------------
        # Evaluate current population
        # -----------------------------------------------------

        obj_costs = evaluate_population(pop, D, T, C)

        archive.update(pop, obj_costs)

        ranks = fast_non_dominated_sort(obj_costs)

        fitness = nsga2_fitness(ranks, obj_costs)

        # -----------------------------------------------------
        # Generate offspring
        # -----------------------------------------------------

        offspring = []

        while len(offspring) < N:

            p1 = tournament_select(pop, fitness, rng)

            p2 = tournament_select(pop, fitness, rng)

            if rng.random() < pc:
                child = order_crossover(p1, p2, rng)
            else:
                child = p1.copy()

            child = swap_mutation(child, pm, rng)

            child = two_opt_fast(
                child,
                M_avg,
                max_passes=2
            )

            # duplicate prevention
            duplicate = False

            for existing in offspring:
                if np.array_equal(child, existing):
                    duplicate = True
                    break

            if not duplicate:
                offspring.append(child)

        offspring = np.array(offspring)

        # -----------------------------------------------------
        # Combine parent + offspring
        # -----------------------------------------------------

        combined_pop = np.vstack([pop, offspring])

        combined_objs = evaluate_population(
            combined_pop,
            D,
            T,
            C
        )

        combined_ranks = fast_non_dominated_sort(
            combined_objs
        )

        # -----------------------------------------------------
        # Survivor selection
        # -----------------------------------------------------

        new_population = []

        for rank in np.unique(combined_ranks):

            idx = np.where(combined_ranks == rank)[0]

            front = combined_pop[idx]

            front_objs = combined_objs[idx]

            if len(new_population) + len(front) <= N:

                new_population.extend(front)

            else:

                cd = crowding_distance(front_objs)

                order = np.argsort(-cd)

                remaining = N - len(new_population)

                selected = front[order[:remaining]]

                new_population.extend(selected)

                break

        pop = np.array(new_population)

        # -----------------------------------------------------
        # Logging
        # -----------------------------------------------------

        unique_routes = len(
            set(tuple(ind) for ind in pop)
        )

        log.append({
            "g": g,
            "archive_size": len(archive),
            "f_best": float(fitness.min()),
            "unique_routes": unique_routes
        })

    return archive.get_objectives().tolist(), log
'''

with open('baselines/nsga2_baseline.py', 'w', encoding='utf-8') as f:
    f.write(nsga2_code)

print('baselines/nsga2_baseline.py rewritten successfully.')

baselines/nsga2_baseline.py rewritten successfully.
